In [1]:
# Cell 1 — Connect to the database
import sqlite3
import pandas as pd
from pathlib import Path

# Database connection — reuse this pattern in every agent
BASE_DIR = Path("../").resolve()
DB_PATH  = str(BASE_DIR / "database" / "delta_disruption.db").replace("\\", "/")

conn = sqlite3.connect(DB_PATH)
print("✓ Connected to delta_disruption.db")

# What tables exist?
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print(f"\nTables available: {tables['name'].tolist()}")

# How many rows in each?
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn)
    print(f"  {table}: {count['n'][0]:,} rows")

✓ Connected to delta_disruption.db

Tables available: ['disrupted_flights', 'passengers']
  disrupted_flights: 75,287 rows
  passengers: 1,974 rows


In [5]:
# Cell 2 — Detection Agent query
# "Give me all critical disruptions from the last scan"

query_detection = """
    SELECT
        FLIGHT_NUMBER,
        ORIGIN_AIRPORT,
        DESTINATION_AIRPORT,
        DEPARTURE_DELAY,
        CANCELLED,
        disruption_type,
        severity
    FROM disrupted_flights
    WHERE severity >= 2
    ORDER BY severity DESC, DEPARTURE_DELAY DESC
    LIMIT 20
"""

df_detected = pd.read_sql(query_detection, conn)
print(f"Critical disruptions found: {len(df_detected)}")
print(f"\n{df_detected.to_string(index=False)}")

Critical disruptions found: 20

 FLIGHT_NUMBER ORIGIN_AIRPORT DESTINATION_AIRPORT  DEPARTURE_DELAY  CANCELLED disruption_type  severity
          1196            OGG                 LAX           1159.0          1    CANCELLATION         5
          1406            LAX                 DTW           1157.0          1    CANCELLATION         5
          2653          13204               12478            910.0          1    CANCELLATION         5
          1150          12173               12892            575.0          1    CANCELLATION         5
          2039            DCA                 ATL            556.0          1    CANCELLATION         5
          2619            ATL                 TYS            530.0          1    CANCELLATION         5
          1501            BOS                 ATL            489.0          1    CANCELLATION         5
           829            BOS                 MSP            428.0          1    CANCELLATION         5
          1496            LGA   

In [7]:
# Cell 3 — Assessment Agent query
# "Score and rank every affected passenger by priority"

query_assessment = """
    SELECT
        p.passenger_id,
        p.flight_id,
        p.loyalty_tier,
        p.special_need,
        p.has_connection,
        p.disruption_type,
        p.severity,
        p.origin,
        p.destination,
        -- Priority score formula
        (p.severity * 10)
        + CASE p.special_need
            WHEN 'MEDICAL'     THEN 30
            WHEN 'WHEELCHAIR'  THEN 20
            WHEN 'INFANT'      THEN 15
            ELSE 0
          END
        + CASE p.loyalty_tier
            WHEN 'DIAMOND'   THEN 25
            WHEN 'PLATINUM'  THEN 20
            WHEN 'GOLD'      THEN 15
            WHEN 'SILVER'    THEN 10
            ELSE 0
          END
        + CASE p.has_connection
            WHEN 1 THEN 20
            ELSE 0
          END
        AS priority_score
    FROM passengers p
    ORDER BY priority_score DESC
"""

df_assessed = pd.read_sql(query_assessment, conn)

print(f"Passengers scored  : {len(df_assessed):,}")
print(f"Highest score      : {df_assessed['priority_score'].max()}")
print(f"Lowest score       : {df_assessed['priority_score'].min()}")
print(f"Average score      : {df_assessed['priority_score'].mean():.1f}")
print(f"\n=== Top 10 Priority Passengers ===")
print(df_assessed.head(10).to_string(index=False))

Passengers scored  : 1,974
Highest score      : 115
Lowest score       : 10
Average score      : 41.1

=== Top 10 Priority Passengers ===
passenger_id flight_id loyalty_tier special_need  has_connection disruption_type  severity origin destination  priority_score
  PAX-0263-0     DL435         GOLD      MEDICAL               1    CANCELLATION         5    JFK         SFO             115
  PAX-0420-0    DL1779         GOLD      MEDICAL               1    CANCELLATION         5    LGA         MSY             115
  PAX-0026-0    DL2599       SILVER      MEDICAL               1    CANCELLATION         5    BNA         ATL             110
  PAX-0084-2    DL2678     PLATINUM   WHEELCHAIR               1    CANCELLATION         5    LGA         BOS             110
  PAX-0261-0    DL1131     PLATINUM   WHEELCHAIR               1    CANCELLATION         5    LGA         FLL             110
  PAX-0335-1    DL1780         GOLD   WHEELCHAIR               1    CANCELLATION         5    BNA         

In [9]:
# Cell 4 — Rebooking Agent query
# "Give me the top priority passengers who need rebooking"

query_rebooking = """
    SELECT
        p.passenger_id,
        p.flight_id,
        p.origin,
        p.destination,
        p.loyalty_tier,
        p.special_need,
        p.has_connection,
        p.disruption_type,
        p.severity,
        (p.severity * 10)
        + CASE p.special_need
            WHEN 'MEDICAL'    THEN 30
            WHEN 'WHEELCHAIR' THEN 20
            WHEN 'INFANT'     THEN 15
            ELSE 0
          END
        + CASE p.loyalty_tier
            WHEN 'DIAMOND'  THEN 25
            WHEN 'PLATINUM' THEN 20
            WHEN 'GOLD'     THEN 15
            WHEN 'SILVER'   THEN 10
            ELSE 0
          END
        + CASE p.has_connection
            WHEN 1 THEN 20
            ELSE 0
          END
        AS priority_score
    FROM passengers p
    WHERE p.disruption_type IN ('CANCELLATION', 'DELAY_CRITICAL', 'DELAY_MAJOR')
    ORDER BY priority_score DESC
"""

df_rebooking = pd.read_sql(query_rebooking, conn)

print(f"Passengers needing rebooking : {len(df_rebooking):,}")
print(f"\n=== Rebooking Queue (Top 15) ===")
print(df_rebooking.head(15).to_string(index=False))

# Summary by disruption type
print(f"\n=== Breakdown by Disruption Type ===")
print(df_rebooking["disruption_type"].value_counts())

Passengers needing rebooking : 1,008

=== Rebooking Queue (Top 15) ===
passenger_id flight_id origin destination loyalty_tier special_need  has_connection disruption_type  severity  priority_score
  PAX-0263-0     DL435    JFK         SFO         GOLD      MEDICAL               1    CANCELLATION         5             115
  PAX-0420-0    DL1779    LGA         MSY         GOLD      MEDICAL               1    CANCELLATION         5             115
  PAX-0026-0    DL2599    BNA         ATL       SILVER      MEDICAL               1    CANCELLATION         5             110
  PAX-0084-2    DL2678    LGA         BOS     PLATINUM   WHEELCHAIR               1    CANCELLATION         5             110
  PAX-0261-0    DL1131    LGA         FLL     PLATINUM   WHEELCHAIR               1    CANCELLATION         5             110
  PAX-0335-1    DL1780    BNA         ATL         GOLD   WHEELCHAIR               1    CANCELLATION         5             105
  PAX-0034-1    DL2489    PHL         DTW      

In [11]:
# Cell 5 — Notification Agent query
# "Generate personalized message data for each passenger"

query_notification = """
    SELECT
        p.passenger_id,
        p.flight_id,
        p.origin,
        p.destination,
        p.disruption_type,
        p.loyalty_tier,
        p.special_need,
        p.has_connection,
        -- Message tone based on loyalty tier
        CASE p.loyalty_tier
            WHEN 'DIAMOND'  THEN 'Dear Valued Diamond Member'
            WHEN 'PLATINUM' THEN 'Dear Valued Platinum Member'
            WHEN 'GOLD'     THEN 'Dear Gold Member'
            WHEN 'SILVER'   THEN 'Dear Silver Member'
            ELSE                 'Dear Passenger'
        END AS salutation,
        -- Message urgency based on disruption
        CASE p.disruption_type
            WHEN 'CANCELLATION'    THEN 'URGENT: Your flight has been cancelled.'
            WHEN 'DELAY_CRITICAL'  THEN 'IMPORTANT: Your flight is delayed over 2 hours.'
            WHEN 'DELAY_MAJOR'     THEN 'UPDATE: Your flight is delayed over 1 hour.'
            ELSE                        'NOTICE: Your flight has a minor delay.'
        END AS message_header,
        -- Special needs flag for agent
        CASE
            WHEN p.special_need != 'NONE' THEN 'Priority assistance required: ' || p.special_need
            ELSE 'Standard handling'
        END AS assistance_note
    FROM passengers p
    ORDER BY p.disruption_type, p.loyalty_tier DESC
"""

df_notifications = pd.read_sql(query_notification, conn)

print(f"Notifications to send : {len(df_notifications):,}")
print(f"\n=== Sample Notifications (First 5) ===")
for _, row in df_notifications.head(5).iterrows():
    print(f"\n--- {row['passenger_id']} | {row['flight_id']} ---")
    print(f"  To       : {row['salutation']}")
    print(f"  Header   : {row['message_header']}")
    print(f"  Route    : {row['origin']} → {row['destination']}")
    print(f"  Assist   : {row['assistance_note']}")

Notifications to send : 1,974

=== Sample Notifications (First 5) ===

--- PAX-0026-0 | DL2599 ---
  To       : Dear Silver Member
  Header   : URGENT: Your flight has been cancelled.
  Route    : BNA → ATL
  Assist   : Priority assistance required: MEDICAL

--- PAX-0026-1 | DL2599 ---
  To       : Dear Silver Member
  Header   : URGENT: Your flight has been cancelled.
  Route    : BNA → ATL
  Assist   : Standard handling

--- PAX-0034-1 | DL2489 ---
  To       : Dear Silver Member
  Header   : URGENT: Your flight has been cancelled.
  Route    : PHL → DTW
  Assist   : Priority assistance required: WHEELCHAIR

--- PAX-0177-1 | DL903 ---
  To       : Dear Silver Member
  Header   : URGENT: Your flight has been cancelled.
  Route    : ATL → BOS
  Assist   : Priority assistance required: MEDICAL

--- PAX-0242-0 | DL2092 ---
  To       : Dear Silver Member
  Header   : URGENT: Your flight has been cancelled.
  Route    : FLL → LGA
  Assist   : Standard handling


In [13]:
# Cell 6 — Always close your connection
conn.close()
print("✓ Database connection closed")

✓ Database connection closed
